In [1]:
import pandas as pd
from typing import List, Union
import itertools

import numpy as np
from collections import defaultdict
import os
import pandas as pd

In [2]:
def estimate_pass_at_k(
    num_samples: Union[int, List[int], np.ndarray],
    num_correct: Union[List[int], np.ndarray],
    k: int
) -> np.ndarray:
    """
    Estimates pass@k of each problem and returns them in an array.
    """

    def estimator(n: int, c: int, k: int) -> float:
        """
        Calculates 1 - comb(n - c, k) / comb(n, k).
        """
        if n - c < k:
            return 1.0
        return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))

    if isinstance(num_samples, int):
        num_samples_it = itertools.repeat(num_samples, len(num_correct))
    else:
        assert len(num_samples) == len(num_correct)
        num_samples_it = iter(num_samples)

    return np.array([estimator(int(n), int(c), k) for n, c in zip(num_samples_it, num_correct)])

In [3]:
# Get list of all files in the directory
files = os.listdir('./CodeQL_Results/')
csv_files = [file for file in files if file.endswith('.csv') and file.startswith('multi-')]
print(csv_files)

['multi-dataset_Qwen_0.0.csv', 'multi-dataset_gemini-2.5-flash_0.8.csv', 'multi-dataset_gpt-4o-mini_0.8.csv', 'multi-dataset_Qwen_0.2.csv', 'multi-dataset_Qwen_0.6.csv', 'multi-dataset_starcoder2_1.0.csv', 'multi-dataset_Qwen_0.4.csv', 'multi-dataset_starcoder2_0.4.csv', 'multi-dataset_Qwen_1.0.csv', 'multi-dataset_starcoder2_0.6.csv', 'multi-dataset_starcoder2_0.2.csv', 'multi-dataset_starcoder2_0.0.csv', 'multi-dataset_gpt-4o-mini_1.0.csv', 'multi-dataset_gemini-2.5-flash_1.0.csv', 'multi-dataset_starcoder2_0.8.csv', 'multi-dataset_gemini-2.5-flash_0.0.csv', 'multi-dataset_gpt-4o-mini_0.2.csv', 'multi-dataset_Qwen_0.8.csv', 'multi-dataset_gpt-4o-mini_0.0.csv', 'multi-dataset_gemini-2.5-flash_0.2.csv', 'multi-dataset_gemini-2.5-flash_0.6.csv', 'multi-dataset_gpt-4o-mini_0.4.csv', 'multi-dataset_gpt-4o-mini_0.6.csv', 'multi-dataset_gemini-2.5-flash_0.4.csv']


In [7]:
final_results = []
vul_count = 0
unique_cwe = []
for file_name in csv_files:
    model_name = "_".join(file_name.split('_')[:-1])
    temp = ".".join(file_name.split('_')[-1].split('.')[:-1])
    print(f"Model Name: {model_name}, Temp: {temp}")
    df = pd.read_csv('./CodeQL_Results/' + file_name)
    results = {}
    cwe_list = {}
    for index, row in df.iterrows():
        
        id = row['id']
        cwe_id = id.split('_')[3]
        language = row['language']
        if language not in results:
            results[language] = defaultdict(list)
            cwe_list[language] = defaultdict(int)
            
        direct_vulnerable = row['direct_vulnerable']
        indirect_vulnerable = row['indirect_vulnerable']
        if direct_vulnerable != 0:
            direct_vulnerable = 1
            vul_count += 1
            cwe_list[language][cwe_id] += 1

             
        if indirect_vulnerable != 0:
            indirect_vulnerable = 1
        results[language][id].append([direct_vulnerable, indirect_vulnerable])
    for language in results.keys():
        current_results = results[language]
        total, correct = [], []
        for result in current_results.values():
            passed = [r[0] for r in result]
            total.append(len(passed))
            correct.append(sum(passed))
        total = np.array(total)
        correct = np.array(correct)
        # print number of non-zero values in correct
        # print((correct != 0).sum())
        unique_cwe.append([model_name,language, temp,len(cwe_list[language].keys()), (correct != 0).sum()])


        ks = [1,3,5]
        vul_at_k = [(estimate_pass_at_k(total, correct, k).mean())*100
                                for k in ks if (total >= k).all()]
        print("vul_at_k", vul_at_k)

        total, correct = [], []
        for result in current_results.values():
            passed = [(r[0] or r[1]) for r in result]
            total.append(len(passed))
            correct.append(sum(passed))
        total = np.array(total)
        correct = np.array(correct)
            # print(total, correct)

        
        ks = [1,3,5]
        in_vul_at_k = [(estimate_pass_at_k(total, correct, k).mean())*100
                                for k in ks if (total >= k).all()]
        print("in_vul_at_k", in_vul_at_k)

        new_security_at_k =[]
        for k in ks:
            total_passed = 0
            for result in current_results.values():
                count = 0
                for i in range(k):
                    if result[i][0] == 0:
                        count += 1
                if count == k:
                    total_passed += 1
            new_security_at_k.append(total_passed/len(current_results.values())*100)
        
        print("new_security_at_k", new_security_at_k)


        in_new_security_at_k =[]
        for k in ks:
            total_passed = 0
            for result in current_results.values():
                count = 0
                for i in range(k):
                    if result[i][0]+result[i][1] == 0:
                        count += 1
                if count == k:
                    total_passed += 1
            in_new_security_at_k.append(total_passed/len(current_results.values())*100)
        
        print("in_new_security_at_k", in_new_security_at_k)

        final_results.append([model_name, language, temp, vul_at_k[0], vul_at_k[1], vul_at_k[2], in_vul_at_k[0], in_vul_at_k[1], in_vul_at_k[2], new_security_at_k[0], new_security_at_k[1], new_security_at_k[2], in_new_security_at_k[0], in_new_security_at_k[1], in_new_security_at_k[2]])



Model Name: multi-dataset_Qwen, Temp: 0.0
vul_at_k [18.30985915492958, 18.30985915492958, 18.30985915492958]
in_vul_at_k [28.169014084507044, 28.169014084507044, 28.169014084507044]
new_security_at_k [81.69014084507043, 81.69014084507043, 81.69014084507043]
in_new_security_at_k [71.83098591549296, 71.83098591549296, 71.83098591549296]
vul_at_k [9.090909090909092, 9.090909090909092, 9.090909090909092]
in_vul_at_k [13.636363636363635, 13.636363636363635, 13.636363636363635]
new_security_at_k [90.9090909090909, 90.9090909090909, 90.9090909090909]
in_new_security_at_k [86.36363636363636, 86.36363636363636, 86.36363636363636]
vul_at_k [9.090909090909092, 9.090909090909092, 9.090909090909092]
in_vul_at_k [15.909090909090908, 15.909090909090908, 15.909090909090908]
new_security_at_k [90.9090909090909, 90.9090909090909, 90.9090909090909]
in_new_security_at_k [84.0909090909091, 84.0909090909091, 84.0909090909091]
vul_at_k [13.043478260869565, 13.043478260869565, 13.043478260869565]
in_vul_at_k 

In [8]:
df = pd.DataFrame(unique_cwe, columns=['Model','Language', 'Temp', 'Unique CWE', 'Vulnerable Prompts'])
df.to_csv('multi-unique_cwe.csv', index=False)

In [9]:
df = pd.DataFrame(final_results, columns=['model', 'Language', 'Temp', 'vul@1','vul@3','vul@5', 'in_vul@1','in_vul@3','in_vul@5', 'security@1','security@3','security@5', 'in_security@1','in_security@3','in_security@5'])
df.to_csv('At_k_Results-Multi.csv', index=False)
